In [1]:
# 在本章学习的基础上创建一个chatbot
import os

from langchain_core.tools import tool
from langchain_classic.tools.render import format_tool_to_openai_tool
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.tools.render import format_tool_to_openai_function
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents.output_parsers.openai_functions import OpenAIFunctionsAgentOutputParser
from langchain_classic.prompts import MessagesPlaceholder
from pydantic import Field, BaseModel

api_key = os.environ.get("DEEPSEEK_API_KEY")
Model = "deepseek-v4-flash"
base_url = "https://api.deepseek.com/"

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=Model,
    base_url=base_url,
    api_key=api_key,
    temperature=0,
    extra_body={
        "think": "disabled"
    }
)

In [3]:
# create tool
@tool
def create_your_own(query: str) -> str:
    """This function can do whatever you would like once you fill it in"""
    print(type(query))
    return query[::-1]

# 创建一个实时获取天气的工具
import requests
import datetime

# define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # 创建请求
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        results = response.json()
    else :
        raise Exception(f"API Request failed with status code: {response.status_code}")


    current_utc_time = datetime.datetime.now(datetime.timezone.utc)
    time_list = [datetime.datetime.fromisoformat(t.replace("Z", "+00:00")).replace(tzinfo=datetime.timezone.utc) for t in results["hourly"]["time"]]
    temperature_list = results['hourly']['temperature_2m']

    closeset_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))

    current_temperature = temperature_list[closeset_time_index]

    return f"The current temperature is {current_temperature} ℃"

# 定义一个链接维基百科工具
import wikipedia
wikipedia.wikipedia.API_URL = "https://en.wikipedia.org/w/api.php"
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries"""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[:3]: # 仅遍历前三个
        try:
            wiki_page = wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page:{page_title}\nSummary:{wiki_page.summary}")
        except(
            wikipedia.exceptions.PageError,
            wikipedia.exceptions.DisambiguationError,
        ):
            pass

    if not  summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [4]:
tools = [get_current_temperature, search_wikipedia, create_your_own]

In [5]:
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.agents.format_scratchpad import format_to_openai_functions
# 导入创建GUI的包
import panel as pn
pn.extension()

import param

class cbfs(param.Parameterized):
    def __init__(self, tools, **params):
        super(cbfs, self).__init__(**params)

        self.panels = []

        self.functions = [
            format_tool_to_openai_function(f)
            for f in tools
        ]

        self.model = ChatOpenAI(temperature=0).bind(
            functions=self.functions
        )

        self.memory = ConversationBufferMemory(
            return_messages=True,
            memory_key="chat_history"
        )

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are helpful but sassy assistant"),
            MessagesPlaceholder(variable_name="chat_history"),
            ("user", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])

        self.chain = RunnablePassthrough.assign(
            agent_scratchpad=lambda x: format_to_openai_functions(
                x["intermediate_steps"]
            )
        ) | self.prompt | self.model | OpenAIFunctionsAgentOutputParser()

        self.qa = AgentExecutor(
            agent=self.chain,
            tools=tools,
            verbose=False,
            memory=self.memory
        )

    def convchain(self, query):
        if not query:
            return

        inp.value = ""

        result = self.qa.invoke({"input": query})
        self.answer = result["output"]

        self.panels.extend([
            pn.Row(
                "User:",
                pn.pane.Markdown(query, width=450)
            ),
            pn.Row(
                "ChatBot:",
                pn.pane.Markdown(
                    self.answer,
                    width=450,
                    styles={"background-color": "#F6F6F6"}
                )
            )
        ])

        return pn.WidgetBox(*self.panels, scroll=True)

In [6]:
cb = cbfs(tools)

inp = pn.widgets.TextInput(placeholder="Enter text here...")

conversation = pn.bind(cb.convchain, inp)

tab1 = pn.Column(
    pn.Row(inp),
    pn.layout.Divider(),
    pn.panel(conversation, loading_indicator=True, height=400),
    pn.layout.Divider(),
)

dashboard = pn.Column(
    pn.Row(pn.pane.Markdown('# QnA_Bot')),
    pn.Tabs(('Conversation', tab1))
)

dashboard

C:\Users\hilary\AppData\Local\Temp\ipykernel_36244\1294108286.py:24: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  self.memory = ConversationBufferMemory(


Column
    [0] Row
        [0] Markdown(str)
    [1] Tabs
        [0] Column
            [0] Row
                [0] TextInput(placeholder='Enter text here...')
            [1] Divider()
            [2] ParamFunction(function, _pane=Str, defer_load=False, height=400, loading_indicator=True)
            [3] Divider()

In [7]:
dashboard

Column
    [0] Row
        [0] Markdown(str)
    [1] Tabs
        [0] Column
            [0] Row
                [0] TextInput(placeholder='Enter text here...')
            [1] Divider()
            [2] ParamFunction(function, _pane=Str, defer_load=False, height=400, loading_indicator=True)
            [3] Divider()